<a href="https://colab.research.google.com/github/JuanRosales707/15_lab_cuellar/blob/develop/LAB_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Instalación de chromadb
!pip install chromadb -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.7/55.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.2/196.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.2/71.2 kB 5.4 MB/s eta 0:

In [3]:
# Importar librerías necesarias
import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions

In [5]:
# Definir motor de embeddings por defecto de ChromaDB (OpenAI-like)
default_ef = embedding_functions.DefaultEmbeddingFunction()

# Initialize ChromaDB client
client = chromadb.Client(Settings(persist_directory="./chroma_db"))

# Get or create collection
collection = client.get_or_create_collection(name="estudiantes", embedding_function=default_ef)

# Simular información de estudiantes
nombres = ["Juan Rosales", "Ana Torres", "Luis Pérez", "Carla Mejía"]
informaciones = [
    "Juan Rosales tiene 18 años, estudia Big Data y Ciencia de Datos en Lima, y le gusta ir al gimnasio.",
    "Ana Torres tiene 20 años, estudia Big Data y Ciencia de Datos en Lima, y le gusta leer novelas.",
    "Luis Pérez tiene 21 años, estudia Big Data y Ciencia de Datos en Lima, y le gusta jugar videojuegos.",
    "Carla Mejía tiene 22 años, estudia Big Data y Ciencia de Datos en Lima, y le gusta hacer yoga."
]

# Agregar datos a la colección
collection.add(
    documents=informaciones,
    ids=["1", "2", "3", "4"],
    metadatas=[{"nombre": n} for n in nombres]
)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 80.5MiB/s]


In [6]:
# Realizar una búsqueda en la base de datos
prompt = "¿Quién practica yoga?"
resultados = collection.query(query_texts=[prompt], n_results=2)

# Mostrar resultados
for doc, meta in zip(resultados['documents'][0], resultados['metadatas'][0]):
    print(f"Resultado encontrado: {meta['nombre']} - {doc}")

Resultado encontrado: Carla Mejía - Carla Mejía tiene 22 años, estudia Big Data y Ciencia de Datos en Lima, y le gusta hacer yoga.
Resultado encontrado: Juan Rosales - Juan Rosales tiene 18 años, estudia Big Data y Ciencia de Datos en Lima, y le gusta ir al gimnasio.


In [7]:
# Visualizar los vectores almacenados
print("Vectores almacenados:")
docs = collection.get()
for i in range(len(docs['ids'])):
    print(f"{docs['ids'][i]}: {docs['documents'][i]}")

Vectores almacenados:
1: Juan Rosales tiene 18 años, estudia Big Data y Ciencia de Datos en Lima, y le gusta ir al gimnasio.
2: Ana Torres tiene 20 años, estudia Big Data y Ciencia de Datos en Lima, y le gusta leer novelas.
3: Luis Pérez tiene 21 años, estudia Big Data y Ciencia de Datos en Lima, y le gusta jugar videojuegos.
4: Carla Mejía tiene 22 años, estudia Big Data y Ciencia de Datos en Lima, y le gusta hacer yoga.


In [9]:
!pip install sentence-transformers -q
from sentence_transformers import SentenceTransformer

In [13]:
# Usamos el modelo 'all-MiniLM-L6-v2'
model = SentenceTransformer('all-MiniLM-L6-v2')

# Crear motor de embeddings personalizado
custom_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# Crear nueva colección con este embedding
collection_v2 = client.get_or_create_collection(name="Tecsup_v2", embedding_function=custom_ef)

# Cargar los mismos datos
collection_v2.add(
    documents=informaciones,
    ids=["1", "2", "3", "4"],
    metadatas=[{"nombre": n} for n in nombres]
)

# Realizar la misma consulta
prompt = "¿Quién practica yoga?"
resultados_v2 = collection_v2.query(query_texts=[prompt], n_results=2)

print("\nResultados con otro motor de embeddings:")
for doc, meta in zip(resultados_v2['documents'][0], resultados_v2['metadatas'][0]):
    print(f"{meta['nombre']} - {doc}")


Resultados con otro motor de embeddings:
Carla Mejía - Carla Mejía tiene 22 años, estudia Big Data y Ciencia de Datos en Lima, y le gusta hacer yoga.
Juan Rosales - Juan Rosales tiene 18 años, estudia Big Data y Ciencia de Datos en Lima, y le gusta ir al gimnasio.
